### Word2Vec based recommender

In [1]:
from utils.ml import data_dir, tokenize, vectorize, load_data, train_test_split
df = load_data(data_dir)
train, test = train_test_split(df)
sequences_train, tokens = tokenize(train)
sequences_test, _ = tokenize(test, test = True, tokens_df=tokens)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/28 17:52:09 WARN Utils: Your hostname, DESKTOP-UQF5BSK, resolves to a loopback address: 127.0.1.1; using 172.24.225.163 instead (on interface eth0)
26/04/28 17:52:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 17:52:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


TOKENIZED DF:


+----------+----+-----+---+-----+-------+--------------------+
|session ID|year|month|day|order|country|            sequence|
+----------+----+-----+---+-----+-------+--------------------+
|         1|2008|    4|  1|    1|     29|[194, 147, 49, 89...|
|         3|2008|    4|  1|    1|     21|[89, 52, 149, 140...|
|         5|2008|    4|  1|    1|      9|               [126]|
|         6|2008|    4|  1|    1|      9|[149, 22, 106, 89...|
|        12|2008|    4|  1|    1|     29|[49, 53, 53, 191,...|
|        13|2008|    4|  1|    1|     38|      [164, 209, 36]|
|        15|2008|    4|  1|    1|     21|[164, 209, 36, 27...|
|        16|2008|    4|  1|    1|     29|           [154, 42]|
|        19|2008|    4|  1|    1|     29|[130, 125, 199, 1...|
|        20|2008|    4|  1|    1|     29|      [188, 107, 84]|
+----------+----+-----+---+-----+-------+--------------------+
only showing top 10 rows
TOKEN INFO: 


+----------------------+-----------------------+------+--------+-----------------+-----+-------+----+---+
|page 1 (main category)|page 2 (clothing model)|colour|location|model photography|price|price 2|page| id|
+----------------------+-----------------------+------+--------+-----------------+-----+-------+----+---+
|                     4|                    P51|     9|       5|                1|   28|      2|   3|  0|
|                     4|                    P57|     4|       1|                1|   43|      1|   4|  1|
|                     2|                    B19|    13|       1|                1|   38|      2|   2|  2|
|                     4|                    P16|     7|       6|                1|   33|      2|   1|  3|
|                     3|                    C34|     7|       6|                1|   48|      1|   2|  4|
|                     1|                    A28|     1|       4|                2|   43|      2|   2|  5|
|                     3|                    C3

+----------+----+-----+---+-----+-------+--------------------+----+
|session ID|year|month|day|order|country|            sequence|next|
+----------+----+-----+---+-----+-------+--------------------+----+
|         4|2008|    4|  1|    1|     21|               [151]| 183|
|         4|2008|    4|  1|    2|     21|          [151, 183]|  52|
|         4|2008|    4|  1|    3|     21|      [151, 183, 52]| 100|
|         8|2008|    4|  1|    1|      9|                [97]|  19|
|         8|2008|    4|  1|    2|      9|            [97, 19]| 199|
|         8|2008|    4|  1|    3|      9|       [97, 19, 199]|  10|
|         8|2008|    4|  1|    4|      9|   [97, 19, 199, 10]|   4|
|         8|2008|    4|  1|    5|      9|[97, 19, 199, 10, 4]| 197|
|         8|2008|    4|  1|    6|      9|[97, 19, 199, 10,...| 136|
|         8|2008|    4|  1|    7|      9|[97, 19, 199, 10,...|  54|
+----------+----+-----+---+-----+-------+--------------------+----+
only showing top 10 rows
TOKEN INFO: 
+---------

### Training word2vec on train set

In [2]:
vocab = vectorize(sequences_train, model_only=True)
test_vectorized = vectorize(sequences_test, return_vocabulary=False)

+---+----------+----+-----+---+-----+-------+----+--------------------+
| id|session ID|year|month|day|order|country|next|              vector|
+---+----------+----+-----+---+-----+-------+----+--------------------+
|151|         4|2008|    4|  1|    1|     21| 183|[-0.1169974431395...|
|151|         4|2008|    4|  1|    2|     21|  52|[-0.1169974431395...|
|183|         4|2008|    4|  1|    2|     21|  52|[-0.1403082609176...|
|151|         4|2008|    4|  1|    3|     21| 100|[-0.1169974431395...|
|183|         4|2008|    4|  1|    3|     21| 100|[-0.1403082609176...|
+---+----------+----+-----+---+-----+-------+----+--------------------+
only showing top 5 rows


In [3]:
test_vectorized.show(20)

+---+----------+----+-----+---+-----+-------+----+--------------------+
| id|session ID|year|month|day|order|country|next|              vector|
+---+----------+----+-----+---+-----+-------+----+--------------------+
|151|         4|2008|    4|  1|    1|     21| 183|[-0.1169974431395...|
|151|         4|2008|    4|  1|    2|     21|  52|[-0.1169974431395...|
|183|         4|2008|    4|  1|    2|     21|  52|[-0.1403082609176...|
|151|         4|2008|    4|  1|    3|     21| 100|[-0.1169974431395...|
|183|         4|2008|    4|  1|    3|     21| 100|[-0.1403082609176...|
| 52|         4|2008|    4|  1|    3|     21| 100|[-0.0065766219049...|
| 97|         8|2008|    4|  1|    1|      9|  19|[0.41288572549819...|
| 97|         8|2008|    4|  1|    2|      9| 199|[0.41288572549819...|
| 19|         8|2008|    4|  1|    2|      9| 199|[0.40424725413322...|
| 97|         8|2008|    4|  1|    3|      9|  10|[0.41288572549819...|
| 19|         8|2008|    4|  1|    3|      9|  10|[0.40424725413

### Weighted pooling

In [4]:
from pyspark.sql import functions as F, Window
from pyspark.ml.stat import Summarizer
decay = 0.8

#Infers original item position in session
w_item = Window.partitionBy("session ID", "id")
test_weighted = (test_vectorized
        .withColumn("item_pos",F.min("order").over(w_item))
        .withColumn("weight",F.pow(F.lit(decay), F.col("order") - F.col("item_pos"))))
sequential_window = Window.partitionBy("session ID", "order")
test_weighted = (test_weighted
        .withColumn("weight_sum", F.sum("weight").over(sequential_window))
        .withColumn("weight", F.col("weight") / F.col("weight_sum")))
pooled_test = test_weighted.groupBy("session ID", "order", "next").agg(Summarizer.sum(F.col("vector"), weightCol=F.col("weight")).alias("pooled"))
pooled_test = pooled_test.join(sequences_test.select("session ID", "order", "sequence"), on = ["session ID", "order"], how = "inner").withColumnRenamed("sequence", "visited")
pooled_test.show()

+----------+-----+----+--------------------+--------------------+
|session ID|order|next|              pooled|             visited|
+----------+-----+----+--------------------+--------------------+
|         4|    1| 183|[-0.1169974431395...|               [151]|
|         4|    2|  52|[-0.1299478974607...|          [151, 183]|
|         4|    3| 100|[-0.0793858992821...|      [151, 183, 52]|
|         8|    1|  19|[0.41288572549819...|                [97]|
|         8|    2| 199|[0.40808657473988...|            [97, 19]|
|         8|    3|  10|[0.23624410435984...|       [97, 19, 199]|
|         8|    4|   4|[0.28266453552173...|   [97, 19, 199, 10]|
|         8|    5| 197|[0.33568871531215...|[97, 19, 199, 10, 4]|
|         8|    6| 136|[0.36603314053599...|[97, 19, 199, 10,...|
|         8|    7|  54|[0.37642743022021...|[97, 19, 199, 10,...|
|         8|    8| 212|[0.39203403834656...|[97, 19, 199, 10,...|
|        10|    1|  27|[-0.1428308635950...|               [164]|
|        1

In [5]:
from pyspark.ml.feature import Normalizer, BucketedRandomProjectionLSH

N = 10
pooled= Normalizer(inputCol="pooled",outputCol="features").transform(pooled_test)
vocab_norm = Normalizer(inputCol="vector", outputCol="features").transform(vocab)
#faster than cosine
lsh = BucketedRandomProjectionLSH(inputCol="features",outputCol="hashes",bucketLength=1,numHashTables=5)
lsh_model = lsh.fit(vocab_norm)


candidates = lsh_model.approxSimilarityJoin(pooled, vocab_norm, threshold=1, distCol="distance")
w = Window.partitionBy(F.col("datasetA.session ID"), F.col("datasetA.order")).orderBy(F.asc("distance"))
top_n = (
    candidates
    .withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") <= N)
    .select(
        F.col("datasetA.session ID").alias("session ID"),
        F.col("datasetA.order").alias("order"),
        F.col("datasetA.next").alias("next"),
        F.col("datasetB.word").alias("word"),
        F.col("distance"),
        F.col("rank")
    )
)

top_n.show(20, truncate=False)
eval_rows = (
    top_n
    .withColumn("hit", (F.col("word") == F.col("next")).cast("int"))
    .withColumn("dcg", F.when(F.col("hit") == 1, 1.0 / F.log2(F.col("rank") + F.lit(1.0))).otherwise(0.0))
)

metrics = (
    eval_rows
    .groupBy("session ID", "order", "next")
    .agg(
        F.max("hit").alias("hit"),
        F.max("dcg").alias("ndcg")
    )
    .agg(
        F.avg("hit").alias(f"recall@{N}"),
        F.avg("ndcg").alias(f"ndcg@{N}")
    )
)

metrics.show()

catalog_size = vocab.select("word").distinct().count()
recommended_items = top_n.select("word").distinct().count()

print(f"Catalog coverage: {recommended_items / catalog_size:.4%} ({recommended_items}/{catalog_size})")

+----------+-----+----+----+------------------+----+
|session ID|order|next|word|distance          |rank|
+----------+-----+----+----+------------------+----+
|8         |3    |10  |97  |0.8024389846074383|1   |
|8         |3    |10  |199 |0.8188373033066817|2   |
|8         |3    |10  |192 |0.8468343300175671|3   |
|8         |3    |10  |100 |0.8706599776476082|4   |
|8         |3    |10  |167 |0.88389536630516  |5   |
|8         |3    |10  |50  |0.8891977658624407|6   |
|8         |3    |10  |137 |0.8953016950618653|7   |
|8         |3    |10  |161 |0.9034445386849169|8   |
|8         |3    |10  |18  |0.9042176902599499|9   |
|8         |3    |10  |193 |0.9051916615315775|10  |
|10        |2    |49  |27  |0.6858808696823764|1   |
|10        |2    |49  |164 |0.7569965896257864|2   |
|10        |2    |49  |61  |0.7610394798013347|3   |
|10        |2    |49  |49  |0.7931168236907449|4   |
|10        |2    |49  |148 |0.8246716436222252|5   |
|10        |2    |49  |105 |0.8476899718460779

+-------------------+-------------------+
|          recall@10|            ndcg@10|
+-------------------+-------------------+
|0.33244985673352434|0.15357772329388303|
+-------------------+-------------------+



Catalog coverage: 100.0000% (213/213)


### Simple Markov (first order) (NOT USED)

In [6]:
import os
#adjust version if fails.
df = load_data(data_dir)
sequences, tokens = tokenize(df, build_sequences=False)

TOKENIZED DF:


+----+-----+---+-----+-------+----------+---+
|year|month|day|order|country|session ID| id|
+----+-----+---+-----+-------+----------+---+
|2008|    4|  1|    1|     29|         1|197|
|2008|    4|  1|    2|     29|         1|149|
|2008|    4|  1|    3|     29|         1| 51|
|2008|    4|  1|    4|     29|         1| 91|
|2008|    4|  1|    5|     29|         1|111|
|2008|    4|  1|    6|     29|         1|193|
|2008|    4|  1|    7|     29|         1| 73|
|2008|    4|  1|    8|     29|         1| 97|
|2008|    4|  1|    9|     29|         1|118|
|2008|    4|  1|    1|     29|         2| 87|
+----+-----+---+-----+-------+----------+---+
only showing top 10 rows
TOKEN INFO: 
+----------------------+-----------------------+------+--------+-----------------+-----+-------+----+---+
|page 1 (main category)|page 2 (clothing model)|colour|location|model photography|price|price 2|page| id|
+----------------------+-----------------------+------+--------+-----------------+-----+-------+----+---+


### Constructing transitions

In [7]:
from pyspark.sql import Window
from pyspark.sql import functions as F
w = Window.partitionBy("session ID").orderBy("order")
transitions = sequences.select(F.col("id").alias("src"), "session ID", "order", F.lag("id", offset = -1).over(w).alias("dst"))
transitions = transitions.dropna()
transitions = transitions.groupBy("src", "dst").count().orderBy("count", ascending = False)
w = Window.partitionBy("src")
transitions = transitions.withColumn("weight", F.col("count") / F.sum("count").over(w)).drop("count")
transitions = transitions.filter(transitions["src"] != transitions["dst"])
print("Total transitions (excluding self-connections): ", transitions.count())
transitions.show(5)

Total transitions (excluding self-connections):  20295
+---+---+--------------------+
|src|dst|              weight|
+---+---+--------------------+
|  0|165| 0.09536784741144415|
|  0|203| 0.05722070844686648|
|  0|126|0.051771117166212535|
|  0|113|0.051771117166212535|
|  0|114|0.035422343324250684|
+---+---+--------------------+
only showing top 5 rows
